In [15]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

DB_PATH = "../data/online_retail.db"
engine = create_engine(f"sqlite:///{DB_PATH}")
print("Setup complete.")

Setup complete.


In [16]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type IN ('table','view')", engine)
print(tables)

                 name
0    raw_transactions
1  clean_transactions
2            rfm_base
3          rfm_scored
4        rfm_segments


In [17]:
rfm = pd.read_sql("SELECT * FROM rfm_segments", engine)

print(f"Loaded {len(rfm):,} customers")
rfm.head()

Loaded 5,939 customers


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segment
0,17641.0,738,1,-6.95,1,1,1,Lost
1,12636.0,738,1,141.00,1,1,1,Lost
2,17592.0,738,2,0.00,1,2,1,Lost
3,17485.0,737,1,-29.55,1,1,1,Lost
4,15833.0,737,1,80.40,1,1,1,Lost


In [18]:
print(rfm['CustomerID'].nunique())

5939


In [19]:
from scipy import stats

rfm['Recency_Z'] = stats.zscore(rfm['Recency'])
rfm['Frequency_Z'] = stats.zscore(rfm['Frequency'])
rfm['Monetary_Z'] = stats.zscore(rfm['Monetary'])

rfm[['CustomerID', 'Recency', 'Recency_Z', 'Frequency', 'Frequency_Z', 'Monetary', 'Monetary_Z']].head()

,CustomerID,Recency,Recency_Z,Frequency,Frequency_Z,Monetary,Monetary_Z
0,17641.0,738,2.532786,1,-0.410358,-6.95,-0.201066
1,12636.0,738,2.532786,1,-0.410358,141.00,-0.190480
2,17592.0,738,2.532786,2,-0.347735,0.00,-0.200569
3,17485.0,737,2.528063,1,-0.410358,-29.55,-0.202683
4,15833.0,737,2.528063,1,-0.410358,80.40,-0.194816


In [20]:
rfm['Outlier'] = (
    (rfm['Recency_Z'].abs() > 3) |
    (rfm['Frequency_Z'].abs() > 3) |
    (rfm['Monetary_Z'].abs() > 3)
)

print(f"Total Customers: {len(rfm):,}")
print(f"Flagged as outliers: {rfm['Outlier'].sum():,}")
print(f"Percentage: {rfm['Outlier'].mean()*100:.2f}%")

Total Customers: 5,939
Flagged as outliers: 70
Percentage: 1.18%


In [21]:
outliers = rfm[rfm['Outlier'] == True]
outliers[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Recency_Z', 'Frequency_Z', 'Monetary_Z']].sort_values('Monetary', ascending=False).head(10)

,CustomerID,Recency,Frequency,Monetary,Recency_Z,Frequency_Z,Monetary_Z
5934,18102.0,0,153,598215.22,-0.953121,9.108405,42.602036
5832,14646.0,1,163,523342.07,-0.948397,9.734639,37.244823
5207,14156.0,9,202,296564.69,-0.910610,12.176953,21.018786
5938,14911.0,0,510,270248.53,-0.953121,31.464972,19.135851
5390,17450.0,7,61,233579.39,-0.920056,3.347048,16.512156
5641,13694.0,3,164,190825.52,-0.938950,9.797262,13.453094
5739,17511.0,2,85,171885.98,-0.943674,4.850011,12.097961
4510,12415.0,23,33,143269.29,-0.844481,1.593592,10.050422
5634,16684.0,3,65,141502.25,-0.938950,3.597542,9.923989
5640,15061.0,3,138,136391.48,-0.938950,8.169053,9.558311


In [22]:
rfm_for_clustering = rfm[rfm['Outlier'] == False].copy()

print(f"Original Customer: {len(rfm):,}")
print(f"Customers for Clustering (Outliers Excluded): {len(rfm_for_clustering):,}")

Original Customer: 5,939
Customers for Clustering (Outliers Excluded): 5,869


In [23]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

scaled_values = scaler.fit_transform(rfm_for_clustering[['Recency', 'Frequency', 'Monetary']])

rfm_for_clustering['Recency_Scaled'] = scaled_values[:, 0]
rfm_for_clustering['Frequency_Scaled'] = scaled_values[:, 1]
rfm_for_clustering['Monetary_Scaled'] = scaled_values[:, 2]

rfm_for_clustering[['Recency', 'Recency_Scaled', 'Frequency', 'Frequency_Scaled', 'Monetary', 'Monetary_Scaled']].head()

,Recency,Recency_Scaled,Frequency,Frequency_Scaled,Monetary,Monetary_Scaled
0,738,1.000000,1,0.000000,-6.95,0.361192
1,738,1.000000,1,0.000000,141.00,0.363321
2,738,1.000000,2,0.018519,0.00,0.361292
3,737,0.998645,1,0.000000,-29.55,0.360867
4,737,0.998645,1,0.000000,80.40,0.362449


In [24]:
from sklearn.cluster import KMeans

inertia_values = []
k_range = range(2,11)

features = rfm_for_clustering[['Recency_Scaled', 'Frequency_Scaled', 'Monetary_Scaled']]

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(features)
    inertia_values.append(kmeans.inertia_)

print(inertia_values)

[217.2220951536201, 139.14991738827123, 89.5868886987329, 68.44701329359735, 49.97780801466682, 43.249635452660286, 38.06555150592405, 33.52771251410759, 30.328776865310747]


In [27]:
kmeans_final = KMeans(n_clusters = 4, random_state=42, n_init=10)
rfm_for_clustering['Cluster'] = kmeans_final.fit_predict(features)

rfm_for_clustering['Cluster'].value_counts().sort_index()


Cluster
0     794
1    3065
2    1484
3     526
Name: count, dtype: int64

In [28]:
cluster_summary = rfm_for_clustering.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean().round(2)
cluster_summary

,Recency,Frequency,Monetary
Cluster,,,
0,602.99,1.87,379.44
1,58.67,5.62,1704.27
2,352.69,3.11,808.68
3,28.70,26.82,9056.71


In [29]:
from sklearn.metrics import silhouette_score

silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(features)
    score = silhouette_score(features, labels)
    silhouette_scores.append(score)
    print(f"k={k}: silhouette score = {score:.4f}")

k=2: silhouette score = 0.6061
k=3: silhouette score = 0.5707
k=4: silhouette score = 0.5513
k=5: silhouette score = 0.5083
k=6: silhouette score = 0.5042
k=7: silhouette score = 0.4780
k=8: silhouette score = 0.4601
k=9: silhouette score = 0.4499
k=10: silhouette score = 0.4173


In [30]:
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score

db_scores = []
ch_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(features)
    db_scores.append(davies_bouldin_score(features, labels))
    ch_scores.append(calinski_harabasz_score(features, labels))

for k, db, ch in zip(k_range, db_scores, ch_scores):
    print(f"k={k}: Davies-Bouldin={db:.4f} (lower better), Calinski-Harabasz={ch:.2f} (higher better)")

k=2: Davies-Bouldin=0.5659 (lower better), Calinski-Harabasz=10979.90 (higher better)
k=3: Davies-Bouldin=0.6409 (lower better), Calinski-Harabasz=10214.32 (higher better)
k=4: Davies-Bouldin=0.6305 (lower better), Calinski-Harabasz=11656.63 (higher better)
k=5: Davies-Bouldin=0.6379 (lower better), Calinski-Harabasz=11893.41 (higher better)
k=6: Davies-Bouldin=0.6482 (lower better), Calinski-Harabasz=13462.01 (higher better)
k=7: Davies-Bouldin=0.6979 (lower better), Calinski-Harabasz=13113.31 (higher better)
k=8: Davies-Bouldin=0.7353 (lower better), Calinski-Harabasz=12882.59 (higher better)
k=9: Davies-Bouldin=0.7384 (lower better), Calinski-Harabasz=12895.04 (higher better)
k=10: Davies-Bouldin=0.8016 (lower better), Calinski-Harabasz=12737.60 (higher better)


In [31]:
kmeans_5 = KMeans(n_clusters=5, random_state=42, n_init=10)
rfm_for_clustering['Cluster_5'] = kmeans_5.fit_predict(features)

cluster_5_summary = rfm_for_clustering.groupby('Cluster_5')[['Recency', 'Frequency', 'Monetary']].mean().round(2)
cluster_5_summary


,Recency,Frequency,Monetary
Cluster_5,,,
0,213.08,3.92,1050.14
1,411.22,2.94,745.64
2,27.84,27.07,9154.96
3,40.29,5.87,1810.30
4,622.60,1.71,320.20


In [32]:
# Finalize k=5 as our chosen clustering
rfm_for_clustering['Cluster'] = rfm_for_clustering['Cluster_5']
rfm_for_clustering = rfm_for_clustering.drop(columns=['Cluster_5'])

# Give the clusters meaningful names based on their RFM profile
cluster_names = {
    2: 'Best Customers',
    3: 'Good Customers',
    0: 'Cooling Off',
    1: 'Lapsing',
    4: 'Lost'
}

rfm_for_clustering['Cluster_Name'] = rfm_for_clustering['Cluster'].map(cluster_names)

rfm_for_clustering[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Cluster', 'Cluster_Name']].head()

,CustomerID,Recency,Frequency,Monetary,Cluster,Cluster_Name
0,17641.0,738,1,-6.95,4,Lost
1,12636.0,738,1,141.00,4,Lost
2,17592.0,738,2,0.00,4,Lost
3,17485.0,737,1,-29.55,4,Lost
4,15833.0,737,1,80.40,4,Lost


In [33]:
pd.crosstab(rfm_for_clustering['Cluster_Name'], rfm_for_clustering['Segment'])

Segment,About to Sleep,At Risk,Cannot Lose Them,Champions,Hibernating,Lost,Loyal Customers,Needs Attention,New Customers,Potential Loyalist,Promising,Uncategorized
Cluster_Name,,,,,,,,,,,,
Best Customers,0,0,7,443,0,0,63,0,0,0,0,0
Cooling Off,294,42,104,0,0,140,190,38,0,0,107,29
Good Customers,0,0,0,828,0,0,913,53,163,405,188,76
Lapsing,339,164,116,0,64,434,0,0,0,0,0,0
Lost,0,78,14,0,48,529,0,0,0,0,0,0
